# 检测未被引用的世界书文件

本Notebook用于检测本格修仙.yaml中未提及的世界书文件。
遍历世界书文件夹，找出YAML中没有引用的文件。

In [26]:
import os
import re
from pathlib import Path

# 设置路径
script_dir = Path.cwd()
cultivation_dir = script_dir.parent  # Cultivation-Card-Game目录
yaml_file = cultivation_dir / '本格修仙.yaml'
worldbook_dir = cultivation_dir / '世界书'

print(f"YAML文件: {yaml_file}")
print(f"世界书目录: {worldbook_dir}")
print("-" * 80)

YAML文件: d:\application\Tavern\settings\git\世界书\Cultivation-Card-Game\本格修仙.yaml
世界书目录: d:\application\Tavern\settings\git\世界书\Cultivation-Card-Game\世界书
--------------------------------------------------------------------------------


In [27]:
# 从YAML文件中提取所有引用的文件
def extract_referenced_files(yaml_content):
    """使用正则表达式匹配 '文件: 世界书\文件名' 的模式
    注意：引用时不会写出扩展名，所以需要处理无扩展名的情况"""
    referenced_files = set()
    
    # 匹配 "文件: 世界书\..." 的模式
    file_pattern = r'文件:\s*世界书[\\\/]([^\n\r]+)'
    
    matches = re.finditer(file_pattern, yaml_content)
    for match in matches:
        filename = match.group(1).strip()
        if filename:
            referenced_files.add(filename)
    
    return referenced_files

# 加载YAML文件
print('正在加载YAML文件...')
with open(yaml_file, 'r', encoding='utf-8') as f:
    yaml_content = f.read()

# 提取引用的文件
print('正在提取YAML中引用的文件...')
referenced_files = extract_referenced_files(yaml_content)
print(f'YAML中引用的文件数: {len(referenced_files)}')

正在加载YAML文件...
正在提取YAML中引用的文件...
YAML中引用的文件数: 96


In [28]:
# 获取世界书文件夹中的所有文件
def get_worldbook_files(worldbook_dir):
    """获取世界书文件夹中的所有文件"""
    files = set()
    
    if not worldbook_dir.exists():
        print(f"错误：世界书文件夹不存在: {worldbook_dir}")
        return files
    
    for entry in worldbook_dir.iterdir():
        if entry.is_file():
            files.add(entry.name)
    
    return files

print('正在扫描世界书文件夹...')
worldbook_files = get_worldbook_files(worldbook_dir)
print(f'世界书文件夹中的文件数: {len(worldbook_files)}')

正在扫描世界书文件夹...
世界书文件夹中的文件数: 96


In [29]:
# 找出未被引用的文件
print("-" * 80)

# 由于YAML中的引用不包含扩展名，需要进行匹配
# 例如：YAML中引用 "五毒教"，实际文件是 "五毒教.txt"
def match_files(referenced_names, actual_files):
    """匹配YAML中的引用名称和实际文件"""
    matched = set()
    unmatched = set()
    
    for ref_name in referenced_names:
        found = False
        for actual_file in actual_files:
            # 获取文件名（不含扩展名）
            file_base = actual_file.rsplit('.', 1)[0] if '.' in actual_file else actual_file
            if file_base == ref_name:
                matched.add(actual_file)
                found = True
                break
        if not found:
            unmatched.add(ref_name)
    
    return matched, unmatched

matched_files, unmatched_refs = match_files(referenced_files, worldbook_files)
unused_files = sorted(worldbook_files - matched_files)

if unused_files:
    print(f"\n✗ 发现 {len(unused_files)} 个未被YAML引用的文件:\n")
    for filename in unused_files:
        print(f"  - {filename}")
else:
    print("\n✓ 所有文件都已在YAML中被引用！")

--------------------------------------------------------------------------------

✓ 所有文件都已在YAML中被引用！


In [30]:
# 验证结果
print("\n" + "=" * 80)
print("详细分析结果:")
print("=" * 80)

print(f"\n【未被YAML引用的文件】({len(unused_files)}):")
for filename in unused_files:
    print(f"  ✗ {filename}")

if unmatched_refs:
    print(f"\n【YAML中引用但不存在的文件】({len(unmatched_refs)}):")
    for ref_name in sorted(unmatched_refs):
        print(f"  ⚠ {ref_name}")
else:
    print(f"\n【YAML中引用但不存在的文件】: 无")


详细分析结果:

【未被YAML引用的文件】(0):

【YAML中引用但不存在的文件】: 无
